# Streaming Responses
We don't need to wait until the entire story is generated. We can present the story to the user as it's being generated.


In [1]:
from openai import OpenAI

openai_client = OpenAI()

In [2]:
# Create a basic stream
messages = [
    {"role": "user", "content": "tell me a bad time story about a unicorn"}
]

stream = openai_client.responses.create(
    model='gpt-4o-mini',
    input=messages,
    stream=True
)

response = None
for event in stream:
    if hasattr(event, 'delta'):
        print(event.delta, end='')
    if hasattr(event, 'response'):
        response = event.response

 # The magic happens here when we iterate over events and print each delta as it arrives.


Once upon a time, in a land where rainbows seemed to touch the ground and flowers bloomed all year round, there was a unicorn named Sparkle. Sparkle was known for her shimmering coat and bright, mischievous eyes, but she had one problem—she was a bit too curious for her own good.

One sunny day, Sparkle ventured into the Enchanted Forest, where she had been warned not to go. The trees whispered secrets and shadows danced, but Sparkle couldn't resist the lure of adventure. As she pranced deeper into the woods, a thick fog rolled in, cloaking her in a blanket of uncertainty.

Suddenly, she stumbled upon a hidden glade, bright with flickering lights. “Fairies!” she exclaimed, her curiosity piqued. The fairies were known to be tricksters, but Sparkle, undeterred, approached them. “Can I play with you?” she asked, her eyes sparkling.

The fairies giggled and beckoned her closer. They tangled her mane with vines and painted her coat with strange colors. “You'll be the most beautiful unicorn!

## Adding Structured Output
We use structured output with our bedtime story example. We need to import from Pydantic and define our model. 
We want: 
- a title 
- a subtitle 
- sections with headers and text



In [3]:
from pydantic import BaseModel, Field

class ArticleSection(BaseModel):
    header: str
    text: str = Field(description="text of the section in markdown")

class ArticleResponse(BaseModel):
    title: str
    subtitle: str
    sections: list[ArticleSection]


In [4]:
# If we try to use responses.parse with stream=True, it doesn't work:

messages = [
    {"role": "user", "content": "tell me a bad time story about a unicorn"}
]

try:
    response = openai_client.responses.parse(
        model="gpt-4o-mini",
        input=messages,
        text_format=ArticleResponse,
        stream=True  # This doesn't work
    )
except Exception as e:
    print(f"Error: {type(e).__name__}: {e}")

# It will fail because the parse method doesn't support streaming. 
# Instead, we need to use responses.stream() with the text_format parameter:


Error: AttributeError: 'str' object has no attribute 'output'


In [5]:
messages = [
    {"role": "user", "content": "tell me a bad time story about a unicorn"}
]

with openai_client.responses.stream(
    model="gpt-4o-mini",
    input=messages,
    text_format=ArticleResponse,
) as stream:
    response = None
    for event in stream:
        if hasattr(event, 'delta'):
            print(event.delta, end='')
        if hasattr(event, 'response'):
            response = event.response


{"title":"The Unicorn That Lost Its Sparkle","subtitle":"A Tale of Mistakes and Lessons Learned","sections":[{"header":"Once upon a time in a mystical forest","text":"there lived a beautiful unicorn named Luna. She was known throughout the land for her radiant sparkle and shimmering mane, a gift that brought joy and magic to everyone who saw her."},{"header":"One fateful day,","text":"Luna, feeling a bit boastful, decided to showcase her sparkle at the Grand Enchantment Festival. Instead of sharing her magic with the forest creatures and helping those in need, she spent her time showing off, dazzling everyone with tricks that spun and twirled."},{"header":"But as she flaunted her brilliance,","text":"a dark cloud appeared overhead, thick and ominous. The more Luna showed off, the more the cloud grew. Soon, it began to rain heavily, and the forest became dark and dreary. The creatures of the forest were unhappy; they felt overshadowed by Luna's arrogance."},{"header":"As the festival co

In [6]:
# We can access the parsed response:

story = response.output_parsed

print('# ' + story.title)
print(story.subtitle)
print()

for section in story.sections:
    print('## ' + section.header)
    print()
    print(section.text)
    print()
    print()


# The Unicorn That Lost Its Sparkle
A Tale of Mistakes and Lessons Learned

## Once upon a time in a mystical forest

there lived a beautiful unicorn named Luna. She was known throughout the land for her radiant sparkle and shimmering mane, a gift that brought joy and magic to everyone who saw her.


## One fateful day,

Luna, feeling a bit boastful, decided to showcase her sparkle at the Grand Enchantment Festival. Instead of sharing her magic with the forest creatures and helping those in need, she spent her time showing off, dazzling everyone with tricks that spun and twirled.


## But as she flaunted her brilliance,

a dark cloud appeared overhead, thick and ominous. The more Luna showed off, the more the cloud grew. Soon, it began to rain heavily, and the forest became dark and dreary. The creatures of the forest were unhappy; they felt overshadowed by Luna's arrogance.


## As the festival continued,

Luna noticed her sparkle fading. Confused, she tried harder to shine, but each 

## Streaming JSON Incrementally
Since the backend is generating JSON (because we specified a schema), we need a way to parse JSON incrementally as it streams.

For XML, there's a parsing approach called SAX that can parse documents as they're being streamed without loading the entire document into memory. We need something similar for JSON.

The library that implements this is called jaxn. Make sure to install it:



In [7]:
!uv add jaxn


Resolved 139 packages in 6ms
Checked 125 packages in 21ms


In [8]:
from jaxn import JSONParserHandler, StreamingJSONParser

The parser uses a handler with callbacks for different events:

- on_field_start(path, field_name) - Called when starting to read a field
- on_field_end(path, field_name, value, parsed_value) - Called when a field is complete
- on_value_chunk(path, field_name, chunk) - Called for each character in string values
- on_array_item_end(path, field_name, item) - Called when an array item is complete

Let's experiment with the parser. First, let's take some raw JSON:

In [9]:
raw_json = """
{"title":"The Misadventures of Sparkle","subtitle":"A Tale of Trouble","sections":[{"header":"Once Upon a Time","text":"In a magical land far away, there lived a young unicorn named Sparkle."},{"header":"The Problem","text":"Sparkle had a mischievous streak that often led to chaos."}]}
""".strip()
raw_json

'{"title":"The Misadventures of Sparkle","subtitle":"A Tale of Trouble","sections":[{"header":"Once Upon a Time","text":"In a magical land far away, there lived a young unicorn named Sparkle."},{"header":"The Problem","text":"Sparkle had a mischievous streak that often led to chaos."}]}'

In [10]:
# Create a handler for our article response
from typing import Any, Dict

class ArticleResponseHandler(JSONParserHandler):

    def on_field_end(self, path: str, field_name: str, value: str, parsed_value: Any = None) -> None:
        if path == '':
            if field_name == 'title':
                print(f'# {value}')
            if field_name == 'subtitle':
                print(value)
        if path == '/sections' and field_name == 'header':
            print(f'\n\n## {value}\n')

    def on_value_chunk(self, path: str, field_name: str, chunk: str) -> None:
        if path == '/sections' and field_name == 'text':
            print(chunk, end='', flush=True)


The handler does three things:

- When the title field ends, print it as an H1 header
- When the subtitle field ends, print it
- When a section header ends, print it as an H2 header
- When section text chunks arrive, print them immediately (streaming the content)

Then create the parser and test it:

In [11]:
handler = ArticleResponseHandler()
parser = StreamingJSONParser(handler=handler)


In [12]:
# And parse the JSON character by character:

for c in raw_json:
    parser.parse_incremental(c)

# We get a nicely formatted story that appears as we parse the JSON. 
# The sections stream character-by-character, which is what we want for a good user experience

# The Misadventures of Sparkle
A Tale of Trouble


## Once Upon a Time

In a magical land far away, there lived a young unicorn named Sparkle.

## The Problem

Sparkle had a mischievous streak that often led to chaos.

## Creating a Streaming Function


Next, we create a function that combines streaming with structured output. I'll take our llm_structured function and modify it:



In [13]:
def llm_structured_stream(
    user_prompt,
    output_type,
    parser_handler=JSONParserHandler(),
    instructions=None,
    model="gpt-4o-mini",
):
    messages = []

    if instructions:
        messages.append({
            "role": "system",
            "content": instructions
        })

    messages.append({
        "role": "user",
        "content": user_prompt
    })

    parser = StreamingJSONParser(handler=parser_handler)

    with openai_client.responses.stream(
        model="gpt-4o-mini",
        input=messages,
        text_format=output_type,
    ) as stream:
        response = None
        for event in stream:
            if hasattr(event, 'delta'):
                parser.parse_incremental(event.delta)
            if hasattr(event, 'response'):
                response = event.response

    return response


This function:

- Builds the messages array with instructions and user prompt
- Creates a StreamingJSONParser with the provided handler
- Uses responses.stream() with text_format parameter
- Parses each delta incrementally through the handler
- Returns the response at the end (for accessing usage metadata)

Let's test it with our article example:



In [14]:
instructions = "your task is to tell the user bad time stories"
user_prompt = "unicorn"

result = llm_structured_stream(
    instructions=instructions,
    user_prompt=user_prompt,
    output_type=ArticleResponse,
    parser_handler=ArticleResponseHandler(),
)

# The story now streams in as it's being generated - the title appears first, then the subtitle, and each section's content streams character by character.

# The Dark Tale of the Lost Unicorn
A Cautionary Tale


## Once Upon a Time

In a mystical land filled with vibrant rainbows and lush green valleys, there lived a unicorn named Wandara. Unlike the others, she was curious and adventurous, always longing to explore beyond the enchanted forest where her kin roamed.

## The Forbidden Forest

One fateful day, driven by her curiosity, Wandara ventured into the Forbidden Forest, a place shunned by all unicorns. It was said that dark magic lingered there, infested by spirits seeking revenge on the pure hearts of unicorns. Ignoring the warnings, Wandara trotted deeper into the shadows.

## A Sinister Encounter

As she wandered further, the sun faded, and the air became thick with gloom. Wandara soon met a shadowy figure—a corrupted, twisted unicorn named Zarren, who once was like her but had succumbed to the darkness of the forest. Zarren, with a mischievous grin, offered her a deal: a way to unleash her hidden powers in exchange for her purity

# Streaming Documentation RAG

In [15]:
# Lets put it all together now.
# First lets get our RAG code

import json

from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import Index

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)
files = reader.read()

parsed_docs = [doc.parse() for doc in files]
chunked_docs = chunk_documents(parsed_docs, size=3000, step=1500)

index = Index(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(chunked_docs)

print(f"Indexed {len(chunked_docs)} chunks from {len(files)} documents")

def search(query):
    results = index.search(
        query=query,
        num_results=5
    )
    return results

instructions = """
You're a documentation assistant. Answer the QUESTION based on the CONTEXT from our documentation.

Use only facts from the CONTEXT when answering.
If the answer isn't in the CONTEXT, say so.
"""

prompt_template = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

def build_prompt(question, search_results):
    context = json.dumps(search_results, indent=2)
    return prompt_template.format(
        question=question,
        context=context
    )

Indexed 382 chunks from 95 documents


In [23]:
# We use our expanded RAGResponse model from the previous lesson:

from typing import Literal

class RAGResponse(BaseModel):
    """
    This model provides a structured answer with metadata about the response,
    including confidence, categorization, and follow-up suggestions.
    """

    answer: str = Field(description="The main answer to the user's question in markdown")
    found_answer: bool = Field(description="True if relevant information was found in the documentation")
    confidence: float = Field(description="Confidence score from 0.0 to 1.0 indicating how certain the answer is")
    confidence_explanation: str = Field(description="Explanation about the confidence level")
    answer_type: Literal["how-to", "explanation", "troubleshooting", "comparison", "reference"] = Field(description="The category of the answer")
    followup_questions: list[str] = Field(description="Suggested follow-up questions the user might want to ask")

# For the handler, we want to stream the answer text character-by-character, 
# and show the metadata when it's complete:

class RAGResponseHandler(JSONParserHandler):
    def on_value_chunk(self, path: str, field_name: str, chunk: str) -> None:
        if path == '' and field_name == 'answer':
            print(chunk, end='', flush=True)

    def on_field_end(self, path: str, field_name: str, value: str, parsed_value: Any = None) -> None:
        if path == '' and field_name == 'answer_type':
            print('\nanswer type:', value)

    def on_array_item_end(self, path: str, field_name: str, item: Dict[str, Any] = None) -> None:
        if field_name == 'followup_questions':
            print('follow up question:', item)

# The handler:

# Streams answer chunks immediately (this is the main content users want to see)
# Prints answer_type when the field completes
# Prints each followup_questions item as it arrives


In [24]:
# Lets create the RAG function that uses the RAGResponseHandler to stream the answer and metadata:
def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    return llm_structured_stream(
        instructions=instructions,
        user_prompt=prompt,
        output_type=RAGResponse,
        parser_handler=RAGResponseHandler()
    )

# And test it:
rag("What is LLM as a judge?")

# The user sees the answer text streaming in real-time, followed by the answer type and follow-up questions. 
# This provides a much better experience than waiting for the entire response.

**LLM as a judge** refers to the use of a Large Language Model (LLM) to evaluate responses according to custom criteria. This approach can be implemented in two main ways:

1. **Reference-based evaluation**: In this method, new responses are compared against a set of approved responses (known as a "ground truth") to determine their correctness. This is useful for regression testing.

2. **Open-ended evaluation**: This method evaluates responses based on specified custom criteria, which is beneficial when approved responses are not available or when context-specific evaluations are needed.

The tutorial outlines how to create an LLM evaluator, focusing on different contexts such as regression testing or prompt comparison. The process includes creating a dataset, running the LLM as a judge, and evaluating the quality of the LLM itself by comparing its evaluations to manual labels.
answer type: explanation
follow up question: What are the steps to create an LLM evaluator?
follow up questi

ParsedResponse(id='resp_0170b6785aaa62c1006adcb63ab7b487d1a91517e81cc3969f', access_programs=None, created_at=1792849467.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-4o-mini-2024-07-18', object='response', output=[ParsedResponseOutputMessage(id='msg_0170b6785aaa62c1006adcb63bdb2c87d18672c43e9cdaf921', content=[ParsedResponseOutputText(annotations=[], text='{"answer":"**LLM as a judge** refers to the use of a Large Language Model (LLM) to evaluate responses according to custom criteria. This approach can be implemented in two main ways:\\n\\n1. **Reference-based evaluation**: In this method, new responses are compared against a set of approved responses (known as a \\"ground truth\\") to determine their correctness. This is useful for regression testing.\\n\\n2. **Open-ended evaluation**: This method evaluates responses based on specified custom criteria, which is beneficial when approved responses are not available or when context-specific evaluati